# Advanced 02 — Integration and Expectation

Practice notebook for the **"Integration and Expectation"** section of the stats course PDF.

## What you will learn

This notebook recaps the theory from the PDF section, then **verifies it with Python**.
Each part ends with a **"Your turn"** prompt; the full **Problem Set** at the end has
**click-to-reveal solutions**.

## How to use

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — statistics is about *relationships*, not memorized formulas.
3. LaTeX uses \( \) for inline math and \[ \] / $$ $$ for display math (KaTeX-friendly).

Let's begin.


---
## Setup — run this first

NumPy for numerics, SciPy for exact distributions, Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy import stats

np.random.seed(42)
rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", scipy.__version__)


---
## Part 1 — The Lebesgue integral as a limit of simple functions

Let \((\Omega,\mathcal{F},P)\) be a probability space and \(X\geq 0\) a random
variable. A **simple function** takes finitely many values:

$$
s(\omega)=\sum_{k=1}^{n}a_k\,\mathbf{1}_{A_k}(\omega),
\qquad a_k\geq 0,\quad A_k\in\mathcal{F}\ \text{disjoint},
$$

and its integral is

$$
\int s\,dP=\sum_{k=1}^{n}a_k\,P(A_k).
$$

For a general \(X\geq 0\) we approximate from below by simple functions
\(s_n\uparrow X\) and define

$$
\int X\,dP=\sup_{n}\int s_n\,dP.
$$

If \(\int|X|\,dP<\infty\) we call \(X\) **integrable** and write
\(E[X]=\int X\,dP\). This single definition subsumes discrete sums
(\(\sum_x x\,P(X=x)\)) and continuous Riemann integrals
(\(\int x\,f(x)\,dx\)).

Below we take \(X\sim\mathrm{Exp}(1)\) (density \(f(x)=e^{-x}\) on
\([0,\infty)\)) and compute \(E[X]=\int x\,f(x)\,dx=1\) three ways:

1. **Simple-function approximation** — \(s_n(x)=\sum_{k} x_k\,\mathbf{1}_{[x_k,x_{k+1})}(x)\)
   with a grid, the very construction above.
2. **Riemann / quadrature** — `scipy.integrate.quad` on \(x\,f(x)\).
3. **Monte Carlo** — \(\frac{1}{N}\sum_{i=1}^{N}X_i\), a Lebesgue sum with
   random atoms \(A_k=\{\text{sample}\ i\}\) of mass \(1/N\).


In [ ]:
from scipy.integrate import quad

# X ~ Exp(1): density f(x) = exp(-x) on [0, inf), true mean = 1.
def f(x):
    return np.exp(-x)

# (1) Simple-function approximation on a grid [0, L] with mesh h.
#     s_n(x) = sum_k x_k * 1_{[x_k, x_{k+1})}(x); integral = sum_k x_k * P(A_k)
#     where P(A_k) = integral of f over [x_k, x_{k+1}).
L, h = 20.0, 0.01
xk = np.arange(0, L, h)
# Use the midpoint of each cell as the simple-function value on that cell.
mid = xk + h / 2
# Mass of each atom = integral of f over the cell, computed exactly via the CDF.
from scipy.stats import expon
cell_mass = expon.cdf(xk + h) - expon.cdf(xk)
simple_integral = np.sum(mid * cell_mass)

# (2) Riemann / quadrature of x * f(x).
riemann_val, riemann_err = quad(lambda x: x * f(x), 0, np.inf)

# (3) Monte Carlo: Lebesgue sum with random atoms of mass 1/N.
rng = np.random.default_rng(42)
N = 200_000
samples = rng.standard_exponential(size=N)  # Exp(1)
mc_val = samples.mean()
mc_std = samples.std(ddof=1) / np.sqrt(N)

print(f"True E[X]              = 1.000000")
print(f"Simple-function approx = {simple_integral:.6f}  (mesh h={h}, L={L})")
print(f"Riemann / quad         = {riemann_val:.6f}  (quad err ~ {riemann_err:.2e})")
print(f"Monte Carlo (N={N})    = {mc_val:.6f} +/- {1.96*mc_std:.4f}")
print()
print("All three converge to the same Lebesgue integral E[X] = 1.")

# Visualize the simple-function approximation against the true density.
fig, ax = plt.subplots(figsize=(9, 4.5))
grid = np.linspace(0, 6, 400)
ax.plot(grid, f(grid), label="true density $f(x)=e^{-x}$", lw=2)
ax.step(xk, f(mid), where="post", alpha=0.6, label="simple $s_n$ (piecewise constant)")
ax.set_xlabel("x"); ax.set_ylabel("density")
ax.set_title(r"Simple-function approximation $s_n \uparrow f$")
ax.legend(); ax.set_ylim(0, 1.05)
plt.tight_layout(); plt.show()


**Your turn:** Halve the mesh `h` (e.g. set `h = 0.005`) and re-run the
simple-function approximation. By roughly what factor does the error
\(|\text{simple\_integral}-1|\) shrink? This is the numerical signature of
\(s_n\uparrow X\) with the integral as the supremum over ever-finer
approximations.


---
## Part 2 — Monotone Convergence Theorem (MCT)

If \(0\leq X_1\leq X_2\leq\dots\) and \(X_n\uparrow X\) almost surely, then

$$
\lim_{n\to\infty}\int X_n\,dP=\int X\,dP.
$$

The integral and the limit can be swapped because nothing "escapes to
infinity" when the sequence only moves *upward*.

We demonstrate this numerically with the standard MCT construction: for a
non-negative \(f\), define

$$
f_n(x)=\min(f(x),\,n)\leq f(x),
\qquad f_n\uparrow f.
$$

Each \(f_n\) is bounded, hence a simple-function limit, and
\(\int f_n\,d\lambda\uparrow\int f\,d\lambda\). We use the heavy-tailed
but integrable \(f(x)=\frac{2}{(1+x)^{3}}\) on \([0,\infty)\), whose
integral is exactly \(1\).


In [ ]:
# f(x) = 2 / (1 + x)^3 on [0, inf): integrable, true integral = 1.
def f_heavy(x):
    return 2.0 / (1.0 + x) ** 3

true_int, _ = quad(f_heavy, 0, np.inf)
print(f"True integral of f over [0, inf) = {true_int:.6f}")

# f_n(x) = min(f(x), n).  Each f_n is bounded (<= n) and f_n ↑ f.
ns = np.array([1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144])
ints_of_fn = np.array([quad(lambda x, n=n: np.minimum(f_heavy(x), n), 0, np.inf)[0]
                       for n in ns])

print(f"{'n':>6} {'int(f_n)':>12} {'gap to int(f)':>14}")
for n, val in zip(ns, ints_of_fn):
    print(f"{n:>6} {val:>12.6f} {true_int - val:>14.2e}")

# The sequence int(f_n) is monotone increasing (MCT) and converges to int(f).
assert np.all(np.diff(ints_of_fn) >= -1e-9), "int(f_n) must be monotone increasing"
print()
print(f"Monotone increasing? {np.all(np.diff(ints_of_fn) > 0)}")
print(f"Final int(f_{{144}}) = {ints_of_fn[-1]:.6f}  (true = {true_int:.6f})")

fig, ax = plt.subplots(figsize=(9, 4.5))
grid = np.linspace(0, 4, 400)
ax.plot(grid, f_heavy(grid), lw=2.5, label="$f(x)=2/(1+x)^3$")
for n in [1, 3, 8, 34]:
    ax.plot(grid, np.minimum(f_heavy(grid), n), ls="--",
            label=f"$f_{{n}}=\\min(f,{n})$")
ax.set_xlabel("x"); ax.set_ylabel("value")
ax.set_title(r"Monotone convergence: $f_n=\min(f,n)\uparrow f$")
ax.legend(); ax.set_ylim(0, 2.2)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(ns, ints_of_fn, "o-", label="$\\int f_n\\,d\\lambda$")
ax.axhline(true_int, color="crimson", ls="--", label="$\\int f\\,d\\lambda=1$")
ax.set_xlabel("n (truncation level)"); ax.set_ylabel("integral")
ax.set_title(r"$\int f_n\,d\lambda\uparrow\int f\,d\lambda$  (MCT)")
ax.legend()
plt.tight_layout(); plt.show()


**Your turn:** Repeat with a *non-integrable* \(f\), e.g. \(f(x)=\frac{1}{1+x}\)
whose integral is \(\infty\). Confirm that \(\int f_n\,d\lambda\) still
increases (MCT holds) but diverges to \(+\infty\) — the theorem is silent on
whether the limit is finite; it just says limit and integral agree.


---
## Part 3 — Dominated Convergence Theorem (DCT)

Let \(X_n\to X\) almost surely. If there is an integrable \(Y\) with
\(|X_n|\leq Y\) almost surely for all \(n\), then \(X\) is integrable and

$$
\lim_{n\to\infty}\int X_n\,dP=\int X\,dP.
$$

The dominator \(Y\) prevents mass from leaking out to infinity, which is the
one failure mode the MCT cannot rule out.

We illustrate DCT with the canonical "truncate the domain" sequence. Let
\(f(x)=e^{-x}\) on \([0,\infty)\) (integrable, integral \(1\)) and set

$$
f_n(x)=f(x)\,\mathbf{1}_{[0,n]}(x)\to f(x)\quad\text{pointwise},
\qquad |f_n|\leq f\in L^1.
$$

Then \(\int f_n\,d\lambda\to\int f\,d\lambda\). We also show what goes
wrong without a dominator: \(g(x)=1\) on \([0,\infty)\), truncated to
\([0,n]\), converges pointwise to \(g\) but \(\int g\,d\lambda=\infty\)
while every \(\int g_n\,d\lambda=n\) is finite — no integrable dominator
exists, and the limit of the integrals (\(\infty\)) does not equal the
integral of the pointwise limit (\(\infty\), but only vacuously; the finite
truncated integrals blow up).


In [ ]:
# DCT example: f(x) = exp(-x) on [0, inf); f_n = f * 1_[0,n]; |f_n| <= f in L^1.
def f_exp(x):
    return np.exp(-x)

true_int_exp, _ = quad(f_exp, 0, np.inf)  # = 1
ns = np.array([1, 2, 4, 8, 16, 32, 64, 128])
ints_exp = np.array([quad(f_exp, 0, n)[0] for n in ns])

print("DCT example (dominator f in L^1):")
print(f"{'n':>6} {'int(f * 1_[0,n])':>18} {'gap to 1':>12}")
for n, val in zip(ns, ints_exp):
    print(f"{n:>6} {val:>18.6f} {true_int_exp - val:>12.2e}")
print(f"Converges to int(f) = {true_int_exp:.6f}  (DCT applies).")

# Counterexample WITHOUT a dominator: g(x) = 1 on [0, inf); g_n = 1_[0,n].
# int(g_n) = n -> infinity, while int(g) = infinity.  No integrable Y dominates.
ints_g = ns.astype(float)  # integral of 1 over [0, n] is exactly n.
print()
print("Counterexample (no L^1 dominator):")
print(f"{'n':>6} {'int(1_[0,n])':>14}")
for n, val in zip(ns, ints_g):
    print(f"{n:>6} {val:>14.1f}")
print("Pointwise limit g = 1, but int(g) = +inf; the truncated integrals diverge.")

# Visualize both.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
grid = np.linspace(0, 4, 400)
for n in [1, 2, 4]:
    axes[0].plot(grid, np.where(grid <= n, f_exp(grid), 0),
                label=f"$f_n=f\\cdot\\mathbf{{1}}_{{[0,{n}]}}$")
axes[0].plot(grid, f_exp(grid), "k--", lw=2, label="$f$ (dominator)")
axes[0].set_title("DCT: $f_n\\to f$, $|f_n|\\leq f\\in L^1$")
axes[0].set_xlabel("x"); axes[0].set_ylabel("value"); axes[0].legend()
axes[0].set_ylim(0, 1.05)

axes[1].semilogy(ns, ints_exp, "o-", color="steelblue", label="$\\int f_n$ (DCT)")
axes[1].axhline(true_int_exp, color="steelblue", ls="--", label="$\\int f = 1$")
axes[1].semilogy(ns, ints_g, "s-", color="crimson", label="$\\int 1_{[0,n]}=n$")
axes[1].set_title("Convergence with vs without a dominator")
axes[1].set_xlabel("n"); axes[1].set_ylabel("integral"); axes[1].legend()
plt.tight_layout(); plt.show()


**Your turn:** Replace \(f(x)=e^{-x}\) with a heavier-tailed but still
integrable \(f(x)=\frac{2}{(1+x)^3}\). Repeat the truncation experiment. How
much larger must \(n\) be to reach a gap below \(10^{-3}\) compared with the
exponential case? Relate this to how fast the dominator's tail mass decays.


---
## Part 4 — Discrete (sum) vs continuous (quad) expectations

The Lebesgue definition \(E[X]=\int X\,dP\) unifies discrete and continuous
random variables. If \(X\) has a pmf \(p\), the integral collapses to a sum
\(\sum_x x\,p(x)\); if \(X\) has a pdf \(f\), it becomes
\(\int x\,f(x)\,dx\), i.e. a Riemann/Lebesgue integral on \(\mathbb{R}\).

We compute \(E[g(X)]\) for the same measurable function \(g\) under three
laws and confirm they all match the abstract integral:

- **Discrete:** \(X\sim\mathrm{Poisson}(\lambda=3)\), sum over the support.
- **Continuous:** \(X\sim\mathrm{Gamma}(2,1)\), integrate with `quad`.
- **Mixed via Monte Carlo:** sample from each law and average \(g(X_i)\).

Take \(g(x)=x^{2}\). For Poisson(\(\lambda\)),
\(E[X^2]=\lambda+\lambda^2=3+9=12\). For Gamma(\(\alpha,\theta=1\)),
\(E[X^2]=\alpha(\alpha+1)=2\cdot3=6\).


In [ ]:
from scipy.stats import poisson, gamma

def g(x):
    return x ** 2

lam = 3.0
alpha = 2.0

# (1) Discrete: sum over the Poisson support (truncate at a safe upper bound).
k = np.arange(0, 50)
pmf = poisson.pmf(k, lam)
disc_exact = np.sum(g(k) * pmf)         # Lebesgue sum over countable atoms
disc_theory = lam + lam ** 2            # Poisson second moment = lambda + lambda^2

# (2) Continuous: integrate x^2 * gamma_pdf(x) over [0, inf) with quad.
def gamma_integrand(x):
    return g(x) * gamma.pdf(x, alpha)
cont_quad, cont_err = quad(gamma_integrand, 0, np.inf)
cont_theory = alpha * (alpha + 1)       # Gamma(a,1) second moment = a(a+1)

# (3) Monte Carlo from each law.
rng = np.random.default_rng(42)
N = 500_000
mc_disc = g(rng.poisson(lam, size=N)).mean()
mc_cont = g(rng.gamma(alpha, size=N)).mean()

print("Discrete  X ~ Poisson(3):")
print(f"  sum    E[g(X)] = {disc_exact:.6f}")
print(f"  theory E[X^2]  = {disc_theory:.6f}")
print(f"  MC     E[g(X)] = {mc_disc:.6f}  (N={N:,})")
print()
print("Continuous X ~ Gamma(2,1):")
print(f"  quad   E[g(X)] = {cont_quad:.6f}  (err ~ {cont_err:.2e})")
print(f"  theory E[X^2]  = {cont_theory:.6f}")
print(f"  MC     E[g(X)] = {mc_cont:.6f}  (N={N:,})")
print()
print("One Lebesgue integral, two regimes, identical agreement within MC error.")

# Compare MC convergence rates: discrete sum vs continuous quad vs Monte Carlo.
fig, ax = plt.subplots(figsize=(9, 4.5))
Ns = np.unique(np.logspace(2, 6, 25).astype(int))
mc_disc_traj = []
mc_cont_traj = []
for Nn in Ns:
    mc_disc_traj.append(g(rng.poisson(lam, size=Nn)).mean())
    mc_cont_traj.append(g(rng.gamma(alpha, size=Nn)).mean())
ax.axhline(disc_theory, color="C0", ls="--", label=f"Poisson theory = {disc_theory}")
ax.axhline(cont_theory, color="C1", ls="--", label=f"Gamma theory = {cont_theory}")
ax.plot(Ns, mc_disc_traj, "o-", color="C0", ms=4, label="Poisson MC")
ax.plot(Ns, mc_cont_traj, "s-", color="C1", ms=4, label="Gamma MC")
ax.set_xscale("log"); ax.set_xlabel("N (Monte Carlo sample size)")
ax.set_ylabel("$E[g(X)]$ estimate")
ax.set_title("Monte Carlo converges to the same Lebesgue integral in both regimes")
ax.legend()
plt.tight_layout(); plt.show()


**Your turn:** Replace \(g(x)=x^{2}\) with \(g(x)=e^{x}\). For the Poisson
law the exact value is \(E[e^{X}]=\exp(\lambda(e-1))\); for Gamma(\(\alpha,1\))
it is \((1-1)^{-\alpha}\)... which blows up. Check numerically: does the
discrete sum still converge, and what does the Gamma Monte Carlo do as
\(N\) grows? Tie the failure to integrability of \(g\) under each law.


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. State the definition of the Lebesgue integral \(\int X\,dP\) for a
non-negative random variable \(X\) via simple-function approximation, and
explain in one sentence why \(E[X]=\int X\,dP\) reduces to the usual sum
\(\sum_x x\,P(X=x)\) when \(X\) is discrete.

2. Let \(f(x)=\frac{2}{(1+x)^{3}}\) on \([0,\infty)\). Define
\(f_n(x)=\min(f(x),n)\). Compute \(\int f_n\,d\lambda\) for \(n=1,2,4\)
and show numerically that the sequence is increasing and converging to
\(\int f\,d\lambda=1\). Which theorem licenses the swap
\(\lim\int f_n=\int\lim f_n\)?

3. Give an example of a sequence \(g_n\to g\) pointwise on \([0,\infty)\)
with \(\int g_n\,d\lambda\) finite for every \(n\) but
\(\lim_n\int g_n\,d\lambda\ne\int g\,d\lambda\). State exactly which
hypothesis of the Dominated Convergence Theorem fails.

4. For \(X\sim\mathrm{Gamma}(2,1)\) compute \(E[X^3]\) three ways:
(i) closed form using the Gamma moments, (ii) `scipy.integrate.quad` on
\(x^{3}f(x)\), (iii) Monte Carlo with a fixed seed and \(N=10^{5}\). Report
all three values and the Monte Carlo standard error.

5. The PDF notes that dominated convergence legitimizes passing the limit
inside expectation for Monte Carlo estimators
\(X_n=\tfrac1n\sum_{i=1}^{n}g(Z_i)\). Identify the dominator \(Y\) and
state precisely where the DCT hypothesis \(|X_n|\leq Y\in L^1\) is used in
the argument that \(\lim_n E[X_n]=E[g(Z_1)]\).


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** A simple function \(s=\sum_k a_k\mathbf{1}_{A_k}\) has integral
\(\int s\,dP=\sum_k a_k\,P(A_k)\). For \(X\geq 0\), choose simple
\(0\leq s_n\uparrow X\) (e.g. dyadic level sets \(A_{n,k}=\{k/2^n\leq
X<(k+1)/2^n\}\)) and define

$$
\int X\,dP=\sup_n\int s_n\,dP=\lim_n\int s_n\,dP.
$$

When \(X\) is discrete, the atoms \(A_k=\{X=x_k\}\) already form a
countable partition of \(\Omega\), so the approximating simple functions are
exact at \(n\geq\)the number of atoms, and the supremum collapses to
\(\sum_k x_k\,P(X=x_k)\).

**2.** \(\int f\,d\lambda=1\) (substitute \(u=1+x\)). For \(n=1,2,4\) we
get (numerically)

```python
from scipy.integrate import quad
import numpy as np
f = lambda x: 2/(1+x)**3
for n in [1, 2, 4]:
    val, _ = quad(lambda x: np.minimum(f(x), n), 0, np.inf)
    print(n, val)
# 1 -> 0.7071, 2 -> 0.9257, 4 -> 0.9961
```

The values are monotone increasing (each \(f_n\leq f_{n+1}\)) and converge to
\(1\). The swap is licensed by the **Monotone Convergence Theorem**, since
\(0\leq f_1\leq f_2\leq\dots\) and \(f_n\uparrow f\).

**3.** \(g_n(x)=\mathbf{1}_{[n,n+1]}(x)\to 0\) pointwise (each \(x\) is
eventually outside the moving unit interval), and \(\int g_n\,d\lambda=1\)
for every \(n\), but \(\int g\,d\lambda=0\). So
\(\lim_n\int g_n=1\ne 0=\int\lim_n g_n\). The DCT hypothesis that fails is
the existence of an integrable dominator \(Y\) with \(|g_n|\leq Y\)
a.s. for all \(n\): the only candidate is \(Y\equiv 1\) on
\(\bigcup_n[n,n+1]=[0,\infty)\), but \(\int 1\,d\lambda=\infty\), so
\(Y\notin L^1\).

**4.** Closed form: for \(X\sim\mathrm{Gamma}(\alpha,1)\),
\(E[X^r]=\alpha(\alpha+1)\cdots(\alpha+r-1)\), so
\(E[X^3]=2\cdot3\cdot4=24\).

```python
import numpy as np
from scipy.integrate import quad
from scipy.stats import gamma
alpha = 2.0
val, err = quad(lambda x: x**3 * gamma.pdf(x, alpha), 0, np.inf)
rng = np.random.default_rng(42)
N = 100_000
x = rng.gamma(alpha, size=N)
mc = np.mean(x**3)
se = np.std(x**3, ddof=1) / np.sqrt(N)
print("quad :", val, err)          # 24.0 ...
print("MC   :", mc, "+/-", 1.96*se)  # ~24 ...
```

All three should agree at \(24\) to within Monte Carlo error.

**5.** Each \(X_n=\frac1n\sum_{i=1}^{n}g(Z_i)\) is bounded by
\(Y=\max_{i\leq n}|g(Z_i)|\) — but that depends on \(n\). The right
dominator for the DCT argument is \(Y=|g(Z_1)|\) itself is *not* a bound on
\(X_n\); instead the standard argument applies DCT to the sequence
\(|X_n-\mu|\) (with \(\mu=E[g(Z_1)]\)) dominated by
\(Y=\tfrac1n\sum_{i=1}^{n}|g(Z_i)-\mu|\), whose expectation is finite because
\(E|g(Z_1)|<\infty\). The hypothesis \(|X_n-\mu|\leq Y\in L^1\) is used
exactly to prevent the averages' deviations from sneaking mass to infinity; it
ultimately rests on the assumption \(E[|g(Z_1)|]<\infty\) stated in the PDF.


</details>
